In [22]:
import tempfile
from unittest import mock
from absl.testing import absltest
from absl.testing import parameterized
import pandas as pd
import numpy as np
import xarray as xr
import tensorflow as tf

# from meridian.planner import roi_to_coefficients_converter
from meridian.data import input_data
from meridian.model import model
from meridian.model import spec
from meridian import constants


In [23]:
class ROIToCoefficientsConverterTest(parameterized.TestCase):
  
  def setUp(self):
    """Set up test fixtures."""
    super().setUp()
    
    # Sample model configuration
    self.model_config = {
      'time_col': 'week',
      'geo_col': 'geo', 
      'population_col': 'population',
      'kpi_type': 'non_revenue',
      'kpi_col': 'conversions',
      'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
      'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
      'media_channels': ['Channel0', 'Channel1', 'Channel2'],
      'reach_cols': ['Channel3_reach'],
      'frequency_cols': ['Channel3_frequency'], 
      'rf_spend_cols': ['Channel3_spend'],
      'rf_channels': ['Channel3']
    }
    
    # Sample ROI data
    self.sample_roi_data = pd.DataFrame({
      'geo': ['Geo0', 'Geo1', 'Geo2'],
      'Channel0': [2.5, 3.2, 2.8],  # ROI values
      'Channel1': [1.8, 2.1, 1.9],
      'Channel2': [3.1, 2.7, 3.4],
      'Channel3': [4.2, 3.8, 4.5]   # RF channel ROI
    })
    
    # Sample parameters data
    self.sample_parameters = pd.DataFrame({
      'MediaVariable': ['Channel0', 'Channel1', 'Channel2', 'Channel3'],
      'Adstock': [0.51, 0.29, 0.17, 0.6],
      'Inflexion': [1.53, 1.23, 1.16, 1.4],
      'Slope': [1.0, 1.0, 1.0, 3.0]
    })
    
    # Create mock InputData object
    self.mock_input_data = self._create_mock_input_data()
    
  def _create_mock_input_data(self):
    """Create a mock InputData object for testing."""
    # Create mock data with proper dimensions
    n_geos, n_times, n_media_channels = 3, 52, 3  # 3 geos, 52 weeks, 3 media channels
    n_rf_channels = 1
    
    # Create mock arrays
    mock_input_data = mock.MagicMock(spec=input_data.InputData)
    
    # Set up geo coordinates
    mock_input_data.geo = xr.DataArray(['Geo0', 'Geo1', 'Geo2'], dims='geo')
    
    # Set up population data
    mock_input_data.population = xr.DataArray([10000, 15000, 12000], dims='geo')
    
    # Set up revenue per kpi (optional)
    mock_input_data.revenue_per_kpi = xr.DataArray(
        np.ones((n_geos, n_times)), 
        dims=['geo', 'time'],
        coords={'geo': ['Geo0', 'Geo1', 'Geo2'], 'time': range(n_times)}
    )
    
    # Set up media spend data
    media_spend_data = np.random.uniform(1000, 5000, (n_geos, n_times, n_media_channels))
    mock_input_data.media_spend = xr.DataArray(
        media_spend_data,
        dims=['geo', 'time', 'media_channel'],
        coords={
            'geo': ['Geo0', 'Geo1', 'Geo2'],
            'time': range(n_times),
            'media_channel': ['Channel0', 'Channel1', 'Channel2']
        }
    )
    
    # Set up RF spend data
    rf_spend_data = np.random.uniform(2000, 8000, (n_geos, n_times, n_rf_channels))
    mock_input_data.rf_spend = xr.DataArray(
        rf_spend_data,
        dims=['geo', 'time', 'rf_channel'],
        coords={
            'geo': ['Geo0', 'Geo1', 'Geo2'],
            'time': range(n_times),
            'rf_channel': ['Channel3']
        }
    )
    
    return mock_input_data

In [18]:
mock_input_data.media_spend

<xarray.DataArray (geo: 3, time: 52, media_channel: 3)> Size: 4kB
array([[[1205.32472418, 3825.74332331, 2481.37281429],
        [1535.85659138, 1173.38902473, 3278.51002555],
        [2401.38299098, 1881.72885196, 4549.33033168],
        [4068.07262184, 4145.1704818 , 2666.15329693],
        [1044.33672258, 4353.10398965, 2642.52291498],
        [2437.41198985, 1441.80729517, 4772.73474103],
        [4158.71173844, 3021.13954827, 2269.6372604 ],
        [1676.97383557, 2734.2457377 , 2777.79878408],
        [1036.56480035, 1684.86364245, 1531.90923069],
        [3121.24197259, 3887.7041383 , 4762.74859941],
        [4633.28197568, 3050.03548387, 3974.1424725 ],
        [2273.2262709 , 3568.89042424, 3123.04533009],
        [4543.71245799, 2533.42906164, 1081.36879374],
        [1068.24356556, 2435.19357764, 1353.93210738],
        [2971.62030212, 2798.90485601, 1410.82410992],
        [2711.9343648 , 1078.32833851, 3628.61907832],
        [2557.19266369, 1055.64126138, 1711.54452014],
        [1483.2390379 , 3932.04621652, 1469.15599996],
        [3620.78149104, 2037.49579742, 1727.46368552],
        [1626.80006227, 1535.46122127, 3990.53937036],
...
        [3667.18695935, 1166.73226253, 1045.84392702],
        [1710.24933922, 4693.73680572, 2619.59966924],
        [4465.75431765, 3712.28624551, 1589.27012013],
        [2933.18415767, 2742.18904635, 2568.05555064],
        [3836.13942138, 2194.80692164, 1553.58533563],
        [2335.6882434 , 3231.84457769, 3061.13515652],
        [4035.96139495, 1394.54146768, 1221.34790961],
        [2078.31702373, 3111.86594127, 2160.6309193 ],
        [4030.43799068, 4826.13012391, 3564.74273541],
        [4408.83815377, 4058.87918301, 2880.54736636],
        [3242.55091818, 4013.01053364, 1119.86666899],
        [1763.26280841, 3035.14550823, 4347.69288375],
        [4504.0680763 , 4099.53143501, 3538.76107988],
        [3050.84367002, 3177.8882511 , 2568.28510376],
        [4327.88396032, 3984.79766246, 2716.48587812],
        [1410.47155476, 4656.90769793, 4097.07946856],
        [3731.08663124, 3130.91931913, 2707.65317513],
        [3298.28128785, 3966.61327324, 4621.07327059],
        [1668.42130543, 1372.53768044, 3054.42333623],
        [4266.54518052, 3739.19659022, 2917.13659038]]])
Coordinates:
  * geo            (geo) <U4 48B 'Geo0' 'Geo1' 'Geo2'
  * time           (time) int64 416B 0 1 2 3 4 5 6 7 ... 44 45 46 47 48 49 50 51
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'